In [1]:
#Instalación
%pip install requests pandas gspread google-auth
%pip install certifi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#Librerías
import certifi
import os
import requests
import pandas as pd
import time
import logging
from datetime import datetime, timedelta
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()
os.environ["SSL_CERT_FILE"] = certifi.where()

#### El ejercicio pedia hacer web scapping del siguiente sitio web: https://www.cvedetails.com , pero en los terminos y condiciones se explica que no se puede scrappear la pagina. La ia de claude recomendo utilizar la API de NVD que tiene la misma información. El ejercicio se realizo utilizando esa fuente de datos.

In [10]:
# Logging 
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
log = logging.getLogger(__name__)

#Configuración 
NVD_API_URL = "https://services.nvd.nist.gov/rest/json/cves/2.0"

# Últimos 3 meses (Enero–Marzo 2026)
DATE_START = "2026-01-01T00:00:00"
DATE_END   = "2026-03-31T23:59:59"
API_KEY = None  

RESULTS_PER_PAGE = 2000   # máximo permitido por la API
DELAY_WITH_KEY   = 0.7    # segundos entre requests con API key
DELAY_NO_KEY     = 6.5    # segundos entre requests sin API key

In [11]:
def build_headers() -> dict:
    headers = {"Content-Type": "application/json"}
    if API_KEY:
        headers["apiKey"] = API_KEY
    return headers


def fetch_cves_page(start_index: int) -> dict | None:
    """Llama a la API del NVD y devuelve el JSON de una página."""
    params = {
        "pubStartDate":   DATE_START,
        "pubEndDate":     DATE_END,
        "resultsPerPage": RESULTS_PER_PAGE,
        "startIndex":     start_index,
    }
    try:
        response = requests.get(
            NVD_API_URL,
            headers=build_headers(),
            params=params,
            timeout=30
        )
        response.raise_for_status()
        return response.json()
    except requests.exceptions.HTTPError as e:
        log.error(f"HTTP Error: {e} | Status: {response.status_code}")
        if response.status_code == 403:
            log.error("API key inválida o rate limit alcanzado.")
        return None
    except requests.exceptions.RequestException as e:
        log.error(f"Error de conexión: {e}")
        return None


def fetch_all_cves() -> list[dict]:
    """Pagina por todos los resultados y devuelve lista completa de CVEs raw."""
    all_raw = []
    start_index = 0
    total_results = None
    delay = DELAY_WITH_KEY if API_KEY else DELAY_NO_KEY

    log.info(f"Fetching CVEs del {DATE_START} al {DATE_END}")

    while True:
        log.info(f"  Página startIndex={start_index} ...")
        data = fetch_cves_page(start_index)

        if data is None:
            log.error("Error en la respuesta. Reintentando en 10s...")
            time.sleep(10)
            data = fetch_cves_page(start_index)
            if data is None:
                log.error("Fallo reintento. Abortando.")
                break

        # Primera vez: loguear el total
        if total_results is None:
            total_results = data.get("totalResults", 0)
            log.info(f"  Total CVEs en el período: {total_results:,}")

        vulnerabilities = data.get("vulnerabilities", [])
        if not vulnerabilities:
            log.info("  Sin más resultados.")
            break

        all_raw.extend(vulnerabilities)
        log.info(f"  Acumulados: {len(all_raw):,} / {total_results:,}")

        start_index += RESULTS_PER_PAGE
        if start_index >= total_results:
            break

        time.sleep(delay)

    return all_raw

In [12]:
# Extracción de datos específicos de cada CVE

def extract_cvss(cve_item: dict) -> tuple[float | None, str]:
    """
    Extrae el CVSS score preferiendo v3.1 > v3.0 > v2.0
    Devuelve (score, version_usada)
    """
    metrics = cve_item.get("metrics", {})

    # CVSS v3.1
    for m in metrics.get("cvssMetricV31", []):
        score = m.get("cvssData", {}).get("baseScore")
        severity = m.get("cvssData", {}).get("baseSeverity", "")
        if score is not None:
            return float(score), "3.1", severity

    # CVSS v3.0
    for m in metrics.get("cvssMetricV30", []):
        score = m.get("cvssData", {}).get("baseScore")
        severity = m.get("cvssData", {}).get("baseSeverity", "")
        if score is not None:
            return float(score), "3.0", severity

    # CVSS v2
    for m in metrics.get("cvssMetricV2", []):
        score = m.get("cvssData", {}).get("baseScore")
        severity = m.get("baseSeverity", "")
        if score is not None:
            return float(score), "2.0", severity

    return None, "N/A", "N/A"


def extract_weaknesses(cve_item: dict) -> str:
    """Extrae el tipo de vulnerabilidad (CWE)."""
    weaknesses = cve_item.get("weaknesses", [])
    cwes = []
    for w in weaknesses:
        for desc in w.get("description", []):
            if desc.get("lang") == "en":
                cwes.append(desc.get("value", ""))
    return " | ".join(cwes) if cwes else "Unknown"


def extract_vendor_product(cve_item: dict) -> tuple[str, str]:
    """
    Extrae vendor y producto de las configuraciones CPE.
    Devuelve los más relevantes (primer vendor/producto encontrado).
    """
    configs = cve_item.get("configurations", [])
    vendors = set()
    products = set()

    for config in configs:
        for node in config.get("nodes", []):
            for match in node.get("cpeMatch", []):
                cpe = match.get("criteria", "")
                # Formato CPE: cpe:2.3:a:VENDOR:PRODUCT:VERSION:...
                parts = cpe.split(":")
                if len(parts) >= 5:
                    vendor = parts[3]
                    product = parts[4]
                    if vendor != "*":
                        vendors.add(vendor)
                    if product != "*":
                        products.add(product)

    return (
        ", ".join(sorted(vendors)[:3]) if vendors else "Unknown",
        ", ".join(sorted(products)[:3]) if products else "Unknown"
    )


def extract_description(cve_item: dict) -> str:
    """Extrae la descripción en inglés."""
    for desc in cve_item.get("descriptions", []):
        if desc.get("lang") == "en":
            text = desc.get("value", "")
            # Truncar a 300 chars para no inflar el CSV
            return text[:300] + "..." if len(text) > 300 else text
    return ""


def parse_cves(raw_list: list[dict]) -> pd.DataFrame:
    """Transforma la lista de CVEs raw del JSON a un DataFrame limpio."""
    rows = []

    for item in raw_list:
        cve = item.get("cve", {})
        cve_id = cve.get("id", "")

        # Fecha de publicación
        pub_date_raw = cve.get("published", "")
        pub_date = pub_date_raw[:10] if pub_date_raw else None  # Solo YYYY-MM-DD

        # CVSS
        cvss_result = extract_cvss(cve)
        cvss_score    = cvss_result[0]
        cvss_version  = cvss_result[1]
        severity      = cvss_result[2]

        # Tipo de vulnerabilidad (CWE)
        vuln_type = extract_weaknesses(cve)

        # Vendor y producto
        vendor, product = extract_vendor_product(cve)

        # Estado
        vuln_status = cve.get("vulnStatus", "")

        # ¿Está en CISA KEV? (vulnerabilidades explotadas activamente)
        in_kev = bool(cve.get("cisaExploitAdd"))

        rows.append({
            "cve_id":       cve_id,
            "pub_date":     pub_date,
            "cvss_score":   cvss_score,
            "cvss_version": cvss_version,
            "severity":     severity,
            "vuln_type":    vuln_type,
            "vendor":       vendor,
            "product":      product,
            "vuln_status":  vuln_status,
            "in_cisa_kev":  in_kev,
            "description":  extract_description(cve),
        })

    df = pd.DataFrame(rows)

    # Tipos de datos
    df["pub_date"]   = pd.to_datetime(df["pub_date"], errors="coerce")
    df["cvss_score"] = pd.to_numeric(df["cvss_score"], errors="coerce")
    df["year_month"] = df["pub_date"].dt.to_period("M").astype(str)

    # Ordenar
    df = df.sort_values("pub_date", ascending=False).reset_index(drop=True)

    log.info(f"DataFrame final: {len(df):,} CVEs")
    return df

In [ ]:
#Exportación 

def save_csv(df: pd.DataFrame, filename: str = "cves_nvd_q1_2026.csv"):
    df_export = df.copy()
    df_export["pub_date"] = df_export["pub_date"].astype(str)
    df_export.to_csv(filename, index=False, encoding="utf-8")
    log.info(f"CSV guardado: {filename}")


def upload_to_google_sheets(df: pd.DataFrame, sheet_name: str = "CVEs_NVD_2026_Q1") -> str | None:
    """
    Sube el DataFrame a Google Sheets para conectar con Looker Studio.

    Setup (una sola vez):
    1. Google Cloud Console → Crear proyecto
    2. Habilitar: Google Sheets API + Google Drive API
    3. IAM → Service Account → Descargar JSON como 'credentials.json'
    4. Crear una Google Sheet → Compartirla con el email de la Service Account
    5. Descomentar la llamada en main()
    """
    try:
        import gspread
        from google.oauth2.service_account import Credentials

        SCOPES = [
            "https://www.googleapis.com/auth/spreadsheets",
            "https://www.googleapis.com/auth/drive",
        ]
        creds = Credentials.from_service_account_file("credentials.json", scopes=SCOPES)
        gc    = gspread.authorize(creds)

        try:
            sh = gc.open(sheet_name)
            ws = sh.sheet1
            ws.clear()
            log.info(f"Sheet '{sheet_name}' encontrada, limpiando...")
        except gspread.SpreadsheetNotFound:
            sh = gc.create(sheet_name)
            ws = sh.sheet1
            log.info(f"Sheet '{sheet_name}' creada.")

        # Preparar: fechas como string, NaN → ""
        df_export = df.copy()
        df_export["pub_date"]   = df_export["pub_date"].astype(str)
        df_export["cvss_score"] = df_export["cvss_score"].fillna("").astype(str)
        df_export               = df_export.fillna("")

        # Subir en batches de 5000 filas para evitar timeouts
        BATCH = 5000
        header = [df_export.columns.tolist()]
        ws.update(header + df_export.iloc[:BATCH].values.tolist())

        for i in range(BATCH, len(df_export), BATCH):
            batch_data = df_export.iloc[i:i+BATCH].values.tolist()
            ws.append_rows(batch_data)
            log.info(f"  Subidas {min(i+BATCH, len(df_export)):,} / {len(df_export):,} filas")
            time.sleep(1)

        log.info(f"{len(df_export):,} filas subidas a Google Sheets")
        log.info(f"URL: {sh.url}")
        return sh.url

    except ImportError:
        log.error("Instalar: pip install gspread google-auth")
    except FileNotFoundError:
        log.error("No se encontró 'credentials.json'. Ver instrucciones en el README.")
    except Exception as e:
        log.error(f"Error Google Sheets: {e}")
    return None

In [ ]:
#Diccionario con categorías de vulnerabilidades basado en CWE (Common Weakness Enumeration)
CWE_CATEGORY_MAP = {
    "CWE-79": "Inyección web", "CWE-352": "Inyección web", "CWE-918": "Inyección web",
    "CWE-80": "Inyección web", "CWE-83": "Inyección web",
    "CWE-89": "Inyección", "CWE-78": "Inyección", "CWE-98": "Inyección",
    "CWE-77": "Inyección", "CWE-88": "Inyección", "CWE-90": "Inyección",
    "CWE-94": "Ejecución remota", "CWE-502": "Ejecución remota", "CWE-434": "Ejecución remota",
    "CWE-862": "Control de acceso", "CWE-863": "Control de acceso",
    "CWE-639": "Control de acceso", "CWE-284": "Control de acceso", "CWE-269": "Control de acceso",
    "CWE-287": "Autenticación", "CWE-306": "Autenticación", "CWE-798": "Autenticación",
    "CWE-384": "Autenticación", "CWE-613": "Autenticación",
    "CWE-200": "Exposición de datos", "CWE-312": "Exposición de datos",
    "CWE-319": "Exposición de datos", "CWE-532": "Exposición de datos",
    "CWE-22": "Filesystem", "CWE-23": "Filesystem", "CWE-36": "Filesystem",
    "CWE-787": "Memoria", "CWE-125": "Memoria", "CWE-416": "Memoria",
    "CWE-476": "Memoria", "CWE-121": "Memoria", "CWE-122": "Memoria",
    "CWE-119": "Memoria", "CWE-415": "Memoria", "CWE-190": "Memoria",
    "CWE-770": "DoS", "CWE-400": "DoS", "CWE-404": "DoS",
    "CWE-326": "Criptografía", "CWE-327": "Criptografía", "CWE-330": "Criptografía",
    "CWE-20": "Validación", "CWE-116": "Validación", "CWE-601": "Validación",
    "CWE-732": "Control de acceso", "CWE-285": "Control de acceso",   
    "CWE-266": "Control de acceso", "CWE-276": "Control de acceso",   
    "CWE-497": "Control de acceso", "CWE-288": "Autenticación",       
    "CWE-295": "Autenticación", "CWE-522": "Autenticación",       
    "CWE-347": "Criptografía", "CWE-59":  "Filesystem",          
    "CWE-73":  "Filesystem",  "CWE-120": "Memoria",             
    "CWE-362": "Memoria", "CWE-401": "Memoria",             
    "CWE-835": "Memoria", "CWE-427": "DoS", "CWE-367": "DoS",  
    "CWE-201": "Exposición de datos",  "CWE-295": "Exposición de datos",          
    "CWE-428": "Validación",  "CWE-266": "Validación",
    "CWE-307":"Autenticación", "CWE-617": "Validación",       
    "CWE-290": "Autenticación",  "CWE-209": "Exposición de datos",   
    "CWE-1321": "Inyección",  "CWE-754": "Validación",         
    "CWE-1284": "Validación", "CWE-74":  "Inyección",          
    "CWE-674": "Memoria", "CWE-346": "Autenticación",      
    "CWE-693": "Control de acceso", "CWE-1333": "DoS",               
    "CWE-345": "Autenticación",  "CWE-1336": "Inyección",       
    "CWE-611": "Inyección",  "CWE-444": "Inyección web",      
    "CWE-204": "Exposición de datos", "CWE-93":  "Inyección",          
    "CWE-451": "Exposición de datos", "CWE-829": "Ejecución remota",
    "CWE-843": "Memoria", "CWE-426": "Ejecución remota",   
    "CWE-129": "Validación", "CWE-250": "Control de acceso",  
    "CWE-667": "Memoria", "CWE-208": "Exposición de datos", 
    "CWE-908": "Exposición de datos", "CWE-822": "Memoria",            
    "CWE-807": "Validación", "CWE-942": "Control de acceso",  
    "CWE-789": "Memoria",  "CWE-126": "Memoria",            
    "CWE-552": "Control de acceso", "CWE-1188": "Control de acceso", 
    "CWE-1021": "Inyección web", "CWE-943": "Inyección",          
    "CWE-598": "Exposición de datos","CWE-644": "Inyección web",      
    "CWE-191": "Memoria",  "CWE-248": "Validación",         
    "CWE-294": "Autenticación", "CWE-203": "Exposición de datos", 
    "CWE-305": "Autenticación", "CWE-1287": "Validación",
    "CWE-1392": "Autenticación", "CWE-193": "Memoria",          
    "CWE-915": "Validación", "CWE-494": "Ejecución remota",   
    "CWE-409": "DoS", "CWE-256": "Exposición de datos", 
    "CWE-35":  "Filesystem",  "CWE-95":  "Ejecución remota",   
    "CWE-640": "Autenticación",  "CWE-321": "Criptografía",       
    "CWE-441": "Ejecución remota", "CWE-369": "Validación",         
    "CWE-280": "Control de acceso", "CWE-1390": "Autenticación",     
    "CWE-134": "Memoria", "CWE-184": "Validación",         
    "CWE-668": "Control de acceso", "CWE-912": "Control de acceso",  
    "CWE-703": "Validación", "CWE-61":  "Filesystem",         
    "CWE-772": "Memoria", "CWE-1260": "Control de acceso", 
    "CWE-538": "Exposición de datos", "CWE-457": "Memoria",
    "CWE-470": "Ejecución remota", "CWE-131": "Memoria", "CWE-185": "Validación",         
    "CWE-1385": "Validación"}

def add_vuln_category(df):
    def get_category(vuln_type):
        if not vuln_type or vuln_type in {"Unknown", "NVD-CWE-noinfo", "NVD-CWE-Other", ""}:
            return "Sin clasificar"
        for cwe in vuln_type.split("|"):
            cwe_clean = cwe.strip().split()[0]
            if cwe_clean in CWE_CATEGORY_MAP:
                return CWE_CATEGORY_MAP[cwe_clean]
        return "Otros"

    df = df.copy()
    df["vuln_category"] = df["vuln_type"].fillna("").apply(get_category)
    return df

In [44]:
# Main 

def main():
    log.info("=" * 60)
    log.info("NVD CVE Fetcher — Enero–Marzo 2026")
    log.info("=" * 60)

    # 1. Fetch de la API
    raw = fetch_all_cves()
    if not raw:
        log.error("No se obtuvieron datos. Verificar conexión o rate limit.")
        return None

    # 2. Parsear a DataFrame
    df = parse_cves(raw)

    # 3. Guardar CSV local
    df = add_vuln_category(df)
    save_csv(df, "cves_nvd_q1_2026_enriched.csv")
    

    #4. Resumen
    print("\n" + "=" * 60)
    print("RESUMEN")
    print("=" * 60)
    print(f"Total CVEs:       {len(df):,}")
    print(f"Período:          {df['pub_date'].min()} → {df['pub_date'].max()}")
    print(f"En CISA KEV:      {df['in_cisa_kev'].sum():,} (explotados activamente)")
    print(f"\nPor mes:")
    print(df.groupby("year_month").size().to_string())
    print(f"\nPor severidad:")
    print(df["severity"].value_counts().to_string())
    print(f"\nTop 10 vendors:")
    print(df["vendor"].value_counts().head(10).to_string())
    print(f"\nTop 10 tipos de vulnerabilidad:")
    print(df["vuln_type"].value_counts().head(10).to_string())
    print("=" * 60)

    return df


if __name__ == "__main__":
    df = main()

2026-04-06 10:00:36,146 [INFO] ============================================================
2026-04-06 10:00:36,148 [INFO] NVD CVE Fetcher — Enero–Marzo 2026
2026-04-06 10:00:36,149 [INFO] ============================================================
2026-04-06 10:00:36,150 [INFO] Fetching CVEs del 2026-01-01T00:00:00 al 2026-03-31T23:59:59
2026-04-06 10:00:36,151 [INFO]   Página startIndex=0 ...
2026-04-06 10:00:38,704 [INFO]   Total CVEs en el período: 16,255
2026-04-06 10:00:38,707 [INFO]   Acumulados: 2,000 / 16,255
2026-04-06 10:00:45,209 [INFO]   Página startIndex=2000 ...
2026-04-06 10:00:47,698 [INFO]   Acumulados: 4,000 / 16,255
2026-04-06 10:00:54,201 [INFO]   Página startIndex=4000 ...
2026-04-06 10:00:57,032 [INFO]   Acumulados: 6,000 / 16,255
2026-04-06 10:01:03,535 [INFO]   Página startIndex=6000 ...
2026-04-06 10:01:05,779 [INFO]   Acumulados: 8,000 / 16,255
2026-04-06 10:01:12,283 [INFO]   Página startIndex=8000 ...
2026-04-06 10:01:15,022 [INFO]   Acumulados: 10,000 / 1


RESUMEN
Total CVEs:       16,255
Período:          2026-01-01 00:00:00 → 2026-03-31 00:00:00
En CISA KEV:      35 (explotados activamente)

Por mes:
year_month
2026-01    5143
2026-02    4808
2026-03    6304

Por severidad:
severity
MEDIUM      6333
HIGH        5658
N/A         2051
CRITICAL    1626
LOW          572
NONE          15

Top 10 vendors:
vendor
Unknown                 7163
linux                    347
microsoft                278
openclaw                 223
tenda                    181
apple                    178
dlink                    127
mozilla                  123
ibm                      116
apple, google, linux     109

Top 10 tipos de vulnerabilidad:
vuln_type
CWE-79     1790
Unknown    1464
CWE-862     874
CWE-89      636
CWE-22      453
CWE-98      374
CWE-918     298
CWE-78      293
CWE-787     287
CWE-352     254


In [ ]:
#Se revisa el df. Hay 2.000 nulos en cvss_score.
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 16255 entries, 0 to 16254
Data columns (total 13 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   cve_id         16255 non-null  str           
 1   pub_date       16255 non-null  datetime64[us]
 2   cvss_score     14204 non-null  float64       
 3   cvss_version   16255 non-null  str           
 4   severity       16255 non-null  str           
 5   vuln_type      16255 non-null  str           
 6   vendor         16255 non-null  str           
 7   product        16255 non-null  str           
 8   vuln_status    16255 non-null  str           
 9   in_cisa_kev    16255 non-null  bool          
 10  description    16255 non-null  str           
 11  year_month     16255 non-null  str           
 12  vuln_category  16255 non-null  str           
dtypes: bool(1), datetime64[us](1), float64(1), str(10)
memory usage: 1.5 MB


In [35]:
df.head()

,cve_id,pub_date,cvss_score,cvss_version,severity,vuln_type,vendor,product,vuln_status,in_cisa_kev,description,year_month,vuln_category
0,CVE-2026-5237,2026-03-31,7.3,3.1,HIGH,CWE-74 | CWE-89,Unknown,Unknown,Awaiting Analysis,False,A security flaw has been discovered in itsourc...,2026-03,Inyección
1,CVE-2026-34155,2026-03-31,5.3,3.1,MEDIUM,CWE-196 | CWE-347,pengutronix,rauc,Analyzed,False,RAUC controls the update process on embedded L...,2026-03,Criptografía
2,CVE-2026-34509,2026-03-31,NaN,N/A,N/A,Unknown,Unknown,Unknown,Rejected,False,Rejected reason: This CVE ID has been rejected...,2026-03,Sin clasificar
3,CVE-2026-3139,2026-03-31,4.3,3.1,MEDIUM,CWE-639,Unknown,Unknown,Awaiting Analysis,False,The User Profile Builder – Beautiful User Regi...,2026-03,Control de acceso
4,CVE-2026-3191,2026-03-31,5.4,3.1,MEDIUM,CWE-352,Unknown,Unknown,Awaiting Analysis,False,The Minify HTML plugin for WordPress is vulner...,2026-03,Inyección web


In [ ]:
#Me quedo con las columnas solicitadas para el ejercicio, mas la categoría de vulnerabilidad.
df_filtrado = df[["cve_id", "pub_date", "cvss_score", "vuln_type", "vuln_category", "vendor", "product"]]
df_filtrado.head()

,cve_id,pub_date,cvss_score,vuln_type,vuln_category,vendor,product
0,CVE-2026-5237,2026-03-31,7.3,CWE-74 | CWE-89,Inyección,Unknown,Unknown
1,CVE-2026-34155,2026-03-31,5.3,CWE-196 | CWE-347,Criptografía,pengutronix,rauc
2,CVE-2026-34509,2026-03-31,NaN,Unknown,Sin clasificar,Unknown,Unknown
3,CVE-2026-3139,2026-03-31,4.3,CWE-639,Control de acceso,Unknown,Unknown
4,CVE-2026-3191,2026-03-31,5.4,CWE-352,Inyección web,Unknown,Unknown


In [ ]:
#La categoria mas recurrente es "Inyeccion web" con un 16% de los casos seguida de "Memoria" con un 15%.
df_filtrado["vuln_category"].value_counts(normalize = True).round(2)*100

vuln_category
Inyección web          16.0
Memoria                15.0
Inyección              13.0
Control de acceso      13.0
Sin clasificar         10.0
Validación              6.0
Autenticación           5.0
Ejecución remota        4.0
Otros                   4.0
Filesystem              4.0
Exposición de datos     4.0
DoS                     3.0
Criptografía            1.0
Name: proportion, dtype: float64

In [ ]:
#Se exporta el df a un csv para el analisis en looker studio.
df_filtrado.to_csv("C:\\Users\\MSI\\OneDrive\\Challenge Meli\\cves_nvd.csv", index=False, encoding="utf-8", ) 
